# Bronze - Extract Raw Metadata

This notebook extracts source data with minimal transformation. It only adds `workspace_id` or parent object ids when needed so downstream notebooks can build keys, nodes, and edges.

In [ ]:
import importlib.util
import subprocess
import sys


def _ensure_pypi_package(module_name: str, package_name: str):
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


try:
    _ensure_pypi_package("sempy", "semantic-link")
    _ensure_pypi_package("sempy_labs", "semantic-link-labs")
except Exception as dep_ex:
    raise RuntimeError(
        "Failed to prepare required semantic-link libraries. "
        "Please ensure the selected Spark environment includes semantic-link and semantic-link-labs. "
        f"Original error: {dep_ex}"
    )

import pandas as pd
import sempy.fabric as fabric
import sempy_labs as labs
import sempy_labs.report as rep
import sempy_labs.lakehouse as labs_lh
import sempy_labs.warehouse as labs_wh
import sempy_labs.directlake as labs_dl
from notebookutils import data as notebook_data
from notebookutils import lakehouse as notebook_lakehouse
from pyspark.sql.types import StringType, StructField, StructType

StatementMeta(, dd093f86-5ee4-4919-805d-03c5b34cf858, 5, Finished, Available, Finished, False)

In [ ]:
import json
from time import perf_counter

if 'targetWorkspaces' not in globals():
    targetWorkspaces = '[]'
if 'targetLakehouseId' not in globals():
    targetLakehouseId = ''
if 'artifactTypes' not in globals():
    artifactTypes = '[]'

def _extract_parameter_candidate(value):
    if not isinstance(value, dict):
        return value

    preferred_keys = [
        'value',
        'content',
        'defaultValue',
        'default',
        'Value',
        'Content',
    ]
    for key in preferred_keys:
        candidate = value.get(key)
        if candidate is not None and str(candidate).strip() != '':
            return candidate

    if len(value) == 1:
        only_value = next(iter(value.values()))
        if only_value is not None and str(only_value).strip() != '':
            return only_value

    return value

def _parse_string_list(candidate_text):
    text = str(candidate_text).strip()
    if not text:
        return []

    try:
        parsed = json.loads(text)
    except Exception:
        # Backward compatibility: older payloads may use single-quoted arrays or comma strings.
        try:
            parsed = json.loads(text.replace("'", '"'))
        except Exception:
            parsed = [segment.strip() for segment in text.strip('[]').split(',') if segment.strip()]

    if isinstance(parsed, dict):
        # Handle parameter wrapper objects recursively.
        return parse_workspace_ids(parsed)

    if isinstance(parsed, list):
        return [str(v).strip().strip("'").strip('"') for v in parsed if str(v).strip()]

    return [str(parsed).strip().strip("'").strip('"')] if str(parsed).strip() else []

def parse_workspace_ids(value):
    candidate = _extract_parameter_candidate(value)
    if isinstance(candidate, list):
        return [str(v).strip() for v in candidate if str(v).strip()]
    if isinstance(candidate, str):
        return _parse_string_list(candidate)
    return []

def parse_artifact_types(value):
    candidate = _extract_parameter_candidate(value)
    if isinstance(candidate, list):
        return [str(v).strip().lower() for v in candidate if str(v).strip()]
    if isinstance(candidate, str):
        return [entry.lower() for entry in _parse_string_list(candidate) if entry]
    return []

WORKSPACE_IDs = parse_workspace_ids(targetWorkspaces)
if not WORKSPACE_IDs:
    # Fallback for runtimes that expose current workspace as a single parameter-like value.
    WORKSPACE_IDs = parse_workspace_ids(globals().get('workspaceId'))
if not WORKSPACE_IDs:
    WORKSPACE_IDs = parse_workspace_ids(globals().get('workspaceID'))
if not WORKSPACE_IDs:
    raise ValueError(
        f"targetWorkspaces is empty or invalid. Received targetWorkspaces={targetWorkspaces!r}. "
        "Provide one or more workspace IDs."
    )

LAKEHOUSE_ID = targetLakehouseId
ARTIFACT_TYPES = parse_artifact_types(artifactTypes)
REQUESTED_ARTIFACTS = set(ARTIFACT_TYPES)

def should_extract(*aliases):
    if not REQUESTED_ARTIFACTS:
        return True
    alias_set = {str(alias).strip().lower() for alias in aliases if str(alias).strip()}
    return bool(alias_set & REQUESTED_ARTIFACTS)

PHASE_TIMERS = {}

def start_phase(name):
    PHASE_TIMERS[name] = perf_counter()

def end_phase(name):
    started_at = PHASE_TIMERS.get(name)
    if started_at is None:
        return
    duration = perf_counter() - started_at
    print(f"[timer] {name}: {duration:.2f}s")

print(f"Processing {len(WORKSPACE_IDs)} workspace(s): {WORKSPACE_IDs}")
print(f"Requested artifact types: {sorted(REQUESTED_ARTIFACTS) if REQUESTED_ARTIFACTS else '[all]'}")

def sanitize_column_names(df):
    if df is None:
        return pd.DataFrame()
    result = df.copy() if hasattr(df, "copy") else pd.DataFrame(df)
    if result.empty and len(result.columns) == 0:
        return pd.DataFrame()
    result.columns = result.columns.str.lower().str.replace(" ", "_").str.replace("-", "_")
    return result

def ensure_columns(df, required_columns):
    if df is None:
        return pd.DataFrame(columns=required_columns)
    result = df.copy() if hasattr(df, "copy") else pd.DataFrame(df)
    if result.empty and len(result.columns) == 0:
        return pd.DataFrame(columns=required_columns)
    for column_name in required_columns:
        if column_name not in result.columns:
            result[column_name] = None
    return result

def normalize_for_spark_write(df, required_columns=None):
    final_df = ensure_columns(df, required_columns or []) if required_columns else (df.copy() if hasattr(df, "copy") else pd.DataFrame(df))
    if len(final_df) == 0 and len(final_df.columns) == 0:
        return pd.DataFrame(columns=required_columns or [])
    for column_name in final_df.columns:
        final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or (isinstance(value, str) and value == "nan") else str(value))
    if required_columns:
        final_df = final_df[required_columns]
    return sanitize_column_names(final_df)

def create_pk(df, columns, pk_name):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    result = df.copy() if hasattr(df, "copy") else pd.DataFrame(df)

    missing_cols = [col for col in columns if col not in result.columns]
    if missing_cols:
        print(f"Warning: Missing columns for primary key '{pk_name}': {missing_cols}")
        print(f"Available columns: {list(result.columns)}")
        result[pk_name] = range(len(result))
        return result

    result[pk_name] = result[columns].apply(
        lambda x: '-'.join(x.dropna().astype(str)),
        axis=1
    )
    return result

def write_raw_table(df, table_name, required_columns=None):
    final_df = normalize_for_spark_write(df, required_columns)
    if final_df.empty and len(final_df.columns) == 0:
        return
    schema = StructType([StructField(column_name, StringType(), True) for column_name in final_df.columns])
    spark.createDataFrame(final_df, schema=schema).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table_name)

StatementMeta(, dd093f86-5ee4-4919-805d-03c5b34cf858, 6, Finished, Available, Finished, False)

Processing 1 workspace(s): ['6feccad2-02d1-4257-b15f-c875f8f7e71b']


In [ ]:
start_phase('fabric_artifacts')
artifact_frames = []
for workspace_id in WORKSPACE_IDs:
    try:
        df_new = fabric.list_items(workspace=workspace_id)
        if not isinstance(df_new, pd.DataFrame):
            df_new = pd.DataFrame(df_new)
        if not df_new.empty:
            artifact_frames.append(df_new)
    except Exception as e:
        print(f"Error processing workspace {workspace_id}: {e}")

df_fabric_artifacts = pd.concat(artifact_frames, axis=0, ignore_index=True) if artifact_frames else pd.DataFrame()
df_fabric_artifacts = sanitize_column_names(df_fabric_artifacts)
write_raw_table(df_fabric_artifacts, 't_fabric_artifacts')
print(f"Created t_fabric_artifacts with {len(df_fabric_artifacts)} rows")
end_phase('fabric_artifacts')

StatementMeta(, dd093f86-5ee4-4919-805d-03c5b34cf858, 7, Finished, Available, Finished, False)

Created t_fabric_artifacts with 13 rows


In [ ]:
semantic_requested = should_extract('semantic_model', 'semanticmodel', 'dataset')
report_requested = should_extract('report')
lakehouse_requested = should_extract('lakehouse')
warehouse_requested = should_extract('warehouse')
directlake_requested = should_extract('directlake', 'direct_lake')

df_semantic_models = (
    df_fabric_artifacts[df_fabric_artifacts['type'] == 'SemanticModel'].copy()
    if 'type' in df_fabric_artifacts.columns and (semantic_requested or directlake_requested)
    else pd.DataFrame()
)
df_reports = (
    df_fabric_artifacts[df_fabric_artifacts['type'] == 'Report'].copy()
    if 'type' in df_fabric_artifacts.columns and report_requested
    else pd.DataFrame()
)
df_lakehouses = (
    df_fabric_artifacts[df_fabric_artifacts['type'] == 'Lakehouse'].copy()
    if 'type' in df_fabric_artifacts.columns and lakehouse_requested
    else pd.DataFrame()
)
df_warehouses = (
    df_fabric_artifacts[df_fabric_artifacts['type'] == 'Warehouse'].copy()
    if 'type' in df_fabric_artifacts.columns and warehouse_requested
    else pd.DataFrame()
)

df_relations = pd.DataFrame()
df_columns = pd.DataFrame()
df_tables = pd.DataFrame()
df_partitions = pd.DataFrame()
df_measures = pd.DataFrame()
df_dependencies = pd.DataFrame()
df_report_metadata = pd.DataFrame()
df_report_pages = pd.DataFrame()
df_report_visuals = pd.DataFrame()
df_report_semantic_objects = pd.DataFrame()
df_lakehouse_tables = pd.DataFrame()
df_lakehouse_columns = pd.DataFrame()
df_warehouse_tables = pd.DataFrame()
df_warehouse_columns = pd.DataFrame()

print(
    f"Enabled sections -> semantic={semantic_requested}, report={report_requested}, "
    f"lakehouse={lakehouse_requested}, warehouse={warehouse_requested}, directlake={directlake_requested}"
)
print(
    f"Input entities -> semantic_models={len(df_semantic_models)}, reports={len(df_reports)}, "
    f"lakehouses={len(df_lakehouses)}, warehouses={len(df_warehouses)}"
)

StatementMeta(, dd093f86-5ee4-4919-805d-03c5b34cf858, 8, Finished, Available, Finished, False)

# Semantic Models

In [ ]:
start_phase('semantic_models')
if semantic_requested:
    relation_frames = []
    column_frames = []
    table_frames = []
    partition_frames = []
    measure_frames = []
    dependency_frames = []

    for row in df_semantic_models[['id', 'workspace_id']].itertuples(index=False):
        dataset_id = row.id
        workspace_id = row.workspace_id

        try:
            df_rel = fabric.list_relationships(dataset=dataset_id, workspace=workspace_id)
            if not isinstance(df_rel, pd.DataFrame):
                df_rel = pd.DataFrame(df_rel)
            if not df_rel.empty:
                df_rel['dataset_id'] = dataset_id
                df_rel['workspace_id'] = workspace_id
                relation_frames.append(create_pk(df_rel, ['Relationship Name', 'dataset_id'], 'relationship_pk'))
        except Exception as e:
            print(f"Error processing relations for {dataset_id} in workspace {workspace_id}: {e}")

        try:
            df_col = fabric.list_columns(dataset=dataset_id, workspace=workspace_id, additional_xmla_properties=['LineageTag'])
            if not isinstance(df_col, pd.DataFrame):
                df_col = pd.DataFrame(df_col)
            if not df_col.empty:
                df_col['dataset_id'] = dataset_id
                df_col['workspace_id'] = workspace_id
                column_frames.append(create_pk(df_col, ['Table Name', 'Column Name', 'dataset_id'], 'column_pk'))
        except Exception as e:
            print(f"Error processing columns for {dataset_id} in workspace {workspace_id}: {e}")

        try:
            df_tbl = fabric.list_tables(dataset=dataset_id, workspace=workspace_id, additional_xmla_properties=['LineageTag'])
            if not isinstance(df_tbl, pd.DataFrame):
                df_tbl = pd.DataFrame(df_tbl)
            if not df_tbl.empty:
                df_tbl['dataset_id'] = dataset_id
                df_tbl['workspace_id'] = workspace_id
                table_frames.append(create_pk(df_tbl, ['Name', 'dataset_id'], 'table_pk'))
        except Exception as e:
            print(f"Error processing tables for {dataset_id} in workspace {workspace_id}: {e}")

        try:
            df_part = fabric.list_partitions(dataset=dataset_id, workspace=workspace_id)
            if not isinstance(df_part, pd.DataFrame):
                df_part = pd.DataFrame(df_part)
            if not df_part.empty:
                df_part['dataset_id'] = dataset_id
                df_part['workspace_id'] = workspace_id
                partition_frames.append(create_pk(df_part, ['Partition Name', 'dataset_id'], 'partition_pk'))
        except Exception as e:
            print(f"Error processing partitions for {dataset_id} in workspace {workspace_id}: {e}")

        try:
            df_meas = fabric.list_measures(dataset=dataset_id, workspace=workspace_id, additional_xmla_properties=['LineageTag'])
            if not isinstance(df_meas, pd.DataFrame):
                df_meas = pd.DataFrame(df_meas)
            if not df_meas.empty:
                df_meas['dataset_id'] = dataset_id
                df_meas['workspace_id'] = workspace_id
                measure_frames.append(create_pk(df_meas, ['Table Name', 'Measure Name', 'dataset_id'], 'measure_pk'))
        except Exception as e:
            print(f"Error processing measures for {dataset_id} in workspace {workspace_id}: {e}")

        try:
            df_dep = labs.get_model_calc_dependencies(dataset_id, workspace_id)
            if not isinstance(df_dep, pd.DataFrame):
                df_dep = pd.DataFrame(df_dep)
            if not df_dep.empty:
                df_dep['dataset_id'] = dataset_id
                df_dep['workspace_id'] = workspace_id
                dependency_frames.append(create_pk(df_dep, ['Full Object Name', 'Referenced Full Object Name', 'dataset_id'], 'dependency_pk'))
        except Exception as e:
            print(f"Error processing dependencies for {dataset_id} in workspace {workspace_id}: {e}")

    df_relations = pd.concat(relation_frames, axis=0, ignore_index=True) if relation_frames else pd.DataFrame()
    df_columns = pd.concat(column_frames, axis=0, ignore_index=True) if column_frames else pd.DataFrame()
    df_tables = pd.concat(table_frames, axis=0, ignore_index=True) if table_frames else pd.DataFrame()
    df_partitions = pd.concat(partition_frames, axis=0, ignore_index=True) if partition_frames else pd.DataFrame()
    df_measures = pd.concat(measure_frames, axis=0, ignore_index=True) if measure_frames else pd.DataFrame()
    df_dependencies = pd.concat(dependency_frames, axis=0, ignore_index=True) if dependency_frames else pd.DataFrame()

    write_raw_table(df_relations, 't_dataset_relations')
    write_raw_table(df_columns, 't_dataset_columns')
    write_raw_table(df_tables, 't_dataset_tables')
    write_raw_table(df_partitions, 't_dataset_partitions')
    write_raw_table(df_measures, 't_dataset_measures')
    write_raw_table(df_dependencies, 't_dataset_dependencies')

    print(
        f"Semantic extraction complete: relations={len(df_relations)}, columns={len(df_columns)}, "
        f"tables={len(df_tables)}, partitions={len(df_partitions)}, measures={len(df_measures)}, dependencies={len(df_dependencies)}"
    )
else:
    print('Skipping semantic model extraction (artifactTypes filter).')
end_phase('semantic_models')

StatementMeta(, 3f03ec58-ec94-4b6f-bf8b-f79ee66945dc, 9, Finished, Available, Finished, False)

# Reports and Report Objects

In [ ]:
def extract_visual_fk(report_source_object):
    if pd.notna(report_source_object) and '[' in str(report_source_object):
        try:
            # Parse "'Page 1'[a4a12234625e46bc8ea0]" -> "Page 1-a4a12234625e46bc8ea0"
            source_str = str(report_source_object)
            parts = source_str.split('[')
            page_name = parts[0].strip("'").strip('"')
            visual_id = parts[1].replace(']', '').replace("'", '').replace('"', '')
            return f"{page_name}-{visual_id}-{report_id}"
        except Exception:
            return None
    return None

start_phase('reports')
if report_requested:
    report_meta_frames = []
    report_page_frames = []
    report_visual_frames = []
    report_semantic_object_frames = []

    for row in df_reports[['id', 'workspace_id']].itertuples(index=False):
        report_id = row.id
        workspace_id = row.workspace_id
        try:
            report = rep.ReportWrapper(report=report_id, workspace=workspace_id)

            df_meta = rep.get_report(report_id, workspace=workspace_id)
            if not isinstance(df_meta, pd.DataFrame):
                df_meta = pd.DataFrame(df_meta)
            if not df_meta.empty:
                df_meta['workspace_id'] = workspace_id
                report_meta_frames.append(create_pk(df_meta, ['Report Id'], 'report_pk'))

            df_pg = report.list_pages()
            if not isinstance(df_pg, pd.DataFrame):
                df_pg = pd.DataFrame(df_pg)
            if not df_pg.empty:
                df_pg['workspace_id'] = workspace_id
                df_pg['report_id'] = report_id
                report_page_frames.append(create_pk(df_pg, ['Page Name', 'report_id'], 'page_pk'))

            df_vis = report.list_visuals()
            if not isinstance(df_vis, pd.DataFrame):
                df_vis = pd.DataFrame(df_vis)
            if not df_vis.empty:
                df_vis['workspace_id'] = workspace_id
                df_vis['report_id'] = report_id
                report_visual_frames.append(create_pk(df_vis, ['Page Display Name', 'Visual Name', 'report_id'], 'visual_pk'))

            df_semObj = report.list_semantic_model_objects(extended=True)
            if not isinstance(df_semObj, pd.DataFrame):
                df_semObj = pd.DataFrame(df_semObj)
            if not df_semObj.empty:
                df_semObj['workspace_id'] = workspace_id
                df_semObj['report_id'] = report_id

                # Create a foreign key to visual_pk in report visuals.
                def create_visual_fk(row_data):
                    if row_data['Report Source'] != 'Page Filter':
                        return extract_visual_fk(row_data['Report Source Object'])
                    return str(row_data['Report Source Object']) + '-' + str(row_data['report_id'])

                df_semObj['visual_fk'] = df_semObj.apply(create_visual_fk, axis=1)
                report_semantic_object_frames.append(
                    create_pk(
                        df_semObj,
                        ['Table Name', 'Object Name', 'Object Type', 'Report Source', 'Report Source Object', 'report_id'],
                        'semantic_object_pk'
                    )
                )
        except Exception as e:
            print(f"Error processing report {report_id} in workspace {workspace_id}: {e}")

    df_report_metadata = pd.concat(report_meta_frames, axis=0, ignore_index=True) if report_meta_frames else pd.DataFrame()
    df_report_pages = pd.concat(report_page_frames, axis=0, ignore_index=True) if report_page_frames else pd.DataFrame()
    df_report_visuals = pd.concat(report_visual_frames, axis=0, ignore_index=True) if report_visual_frames else pd.DataFrame()
    df_report_semantic_objects = pd.concat(report_semantic_object_frames, axis=0, ignore_index=True) if report_semantic_object_frames else pd.DataFrame()

    write_raw_table(df_report_metadata, 't_report_metadata')
    write_raw_table(df_report_pages, 't_report_pages')
    write_raw_table(df_report_visuals, 't_report_visuals')
    write_raw_table(df_report_semantic_objects, 't_report_semantic_objects')

    print(
        f"Report extraction complete: metadata={len(df_report_metadata)}, pages={len(df_report_pages)}, "
        f"visuals={len(df_report_visuals)}, semantic_objects={len(df_report_semantic_objects)}"
    )
else:
    print('Skipping report extraction (artifactTypes filter).')
end_phase('reports')

StatementMeta(, 3f03ec58-ec94-4b6f-bf8b-f79ee66945dc, 10, Finished, Available, Finished, False)

/tmp/ipykernel_6721/1278872090.py:41: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or (isinstance(value, str) and value == "nan") else str(value))


/tmp/ipykernel_6721/1278872090.py:41: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or (isinstance(value, str) and value == "nan") else str(value))


/tmp/ipykernel_6721/1278872090.py:41: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or (isinstance(value, str) and value == "nan") else str(value))


/tmp/ipykernel_6721/1278872090.py:41: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or (isinstance(value, str) and value == "nan") else str(value))


/tmp/ipykernel_6721/1278872090.py:41: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or (isinstance(value, str) and value == "nan") else str(value))


In [ ]:
start_phase('lakehouse_metadata')
if lakehouse_requested:
    df_lakehouses_meta_frames = []
    for workspace_id in WORKSPACE_IDs:
        try:
            df_lh_new = labs_lh.list_lakehouses(workspace=workspace_id)
            if not isinstance(df_lh_new, pd.DataFrame):
                df_lh_new = pd.DataFrame(df_lh_new)
            if not df_lh_new.empty:
                df_lh_new['workspace_id'] = workspace_id
                df_lakehouses_meta_frames.append(df_lh_new)
        except Exception as e:
            print(f"Error processing workspace {workspace_id}: {e}")

    df_lakehouses_meta = pd.concat(df_lakehouses_meta_frames, axis=0, ignore_index=True) if df_lakehouses_meta_frames else pd.DataFrame()
    df_lakehouses_meta = sanitize_column_names(df_lakehouses_meta)
    write_raw_table(df_lakehouses_meta, 't_lakehouses_meta')
else:
    print('Skipping lakehouse metadata extraction (artifactTypes filter).')
end_phase('lakehouse_metadata')

StatementMeta(, 3f03ec58-ec94-4b6f-bf8b-f79ee66945dc, 11, Finished, Available, Finished, False)

In [ ]:
start_phase('warehouse_metadata')
if warehouse_requested:
    df_warehouses_meta_frames = []
    for workspace_id in WORKSPACE_IDs:
        try:
            df_wh_new = labs_wh.list_warehouses(workspace=workspace_id)
            if not isinstance(df_wh_new, pd.DataFrame):
                df_wh_new = pd.DataFrame(df_wh_new)
            if not df_wh_new.empty:
                df_wh_new['workspace_id'] = workspace_id
                df_warehouses_meta_frames.append(df_wh_new)
        except Exception as e:
            print(f"Error processing workspace {workspace_id}: {e}")

    df_warehouses_meta = pd.concat(df_warehouses_meta_frames, axis=0, ignore_index=True) if df_warehouses_meta_frames else pd.DataFrame()
    df_warehouses_meta = sanitize_column_names(df_warehouses_meta)
    write_raw_table(df_warehouses_meta, 't_warehouses_meta')
else:
    print('Skipping warehouse metadata extraction (artifactTypes filter).')
end_phase('warehouse_metadata')

StatementMeta(, 3f03ec58-ec94-4b6f-bf8b-f79ee66945dc, 12, Finished, Available, Finished, False)

In [ ]:
df_lakehouse_tables = pd.DataFrame()
df_lakehouse_columns = pd.DataFrame()
df_lakehouse_shortcuts = pd.DataFrame()

start_phase('lakehouse_structure')
if lakehouse_requested:
    lakehouse_table_frames = []
    lakehouse_column_frames = []
    lakehouse_shortcut_frames = []

    for row in df_lakehouses[['id', 'workspace_id', 'display_name']].itertuples(index=False):
        lakehouse_id = row.id
        workspace_id = row.workspace_id
        try:
            df_tbl = labs.lakehouse.get_lakehouse_tables(lakehouse=lakehouse_id, workspace=workspace_id)
            if not isinstance(df_tbl, pd.DataFrame):
                df_tbl = pd.DataFrame(df_tbl)
            if not df_tbl.empty:
                df_tbl['lakehouse_id'] = lakehouse_id
                df_tbl['workspace_id'] = workspace_id
                lakehouse_table_frames.append(create_pk(df_tbl, ['Table Name', 'lakehouse_id'], 'lakehouse_table_pk'))

            df_col = labs.lakehouse.get_lakehouse_columns(lakehouse=lakehouse_id, workspace=workspace_id)
            if not isinstance(df_col, pd.DataFrame):
                df_col = pd.DataFrame(df_col)
            if not df_col.empty:
                df_col['lakehouse_id'] = lakehouse_id
                df_col['workspace_id'] = workspace_id
                lakehouse_column_frames.append(create_pk(df_col, ['Table Name', 'Column Name', 'lakehouse_id'], 'lakehouse_column_pk'))
        except Exception as e:
            print(f"Error processing lakehouse {lakehouse_id} in workspace {workspace_id}: {e}")

        try:
            df_shortcuts = labs.lakehouse.list_shortcuts(lakehouse=lakehouse_id, workspace=workspace_id)
            if not isinstance(df_shortcuts, pd.DataFrame):
                df_shortcuts = pd.DataFrame(df_shortcuts)
            if not df_shortcuts.empty:
                df_shortcuts['lakehouse_id'] = lakehouse_id
                df_shortcuts['workspace_id'] = workspace_id
                lakehouse_shortcut_frames.append(create_pk(df_shortcuts, ['Shortcut Name', 'lakehouse_id'], 'lakehouse_shortcut_pk'))
        except Exception as e:
            print(f"Error processing lakehouse shortcuts for {lakehouse_id} in workspace {workspace_id}: {e}")

    df_lakehouse_tables = pd.concat(lakehouse_table_frames, axis=0, ignore_index=True) if lakehouse_table_frames else pd.DataFrame()
    df_lakehouse_columns = pd.concat(lakehouse_column_frames, axis=0, ignore_index=True) if lakehouse_column_frames else pd.DataFrame()
    df_lakehouse_shortcuts = pd.concat(lakehouse_shortcut_frames, axis=0, ignore_index=True) if lakehouse_shortcut_frames else pd.DataFrame()

    write_raw_table(df_lakehouse_tables, 't_lakehouse_tables')
    write_raw_table(df_lakehouse_columns, 't_lakehouse_columns')
    write_raw_table(df_lakehouse_shortcuts, 't_lakehouse_shortcuts')
else:
    print('Skipping lakehouse table/column/shortcut extraction (artifactTypes filter).')
end_phase('lakehouse_structure')

df_warehouse_tables = pd.DataFrame()
df_warehouse_columns = pd.DataFrame()

start_phase('warehouse_structure')
if warehouse_requested:
    warehouse_table_frames = []
    warehouse_column_frames = []

    for row in df_warehouses[['id', 'workspace_id', 'display_name']].itertuples(index=False):
        warehouse_id = row.id
        workspace_id = row.workspace_id
        warehouse_name = row.display_name if pd.notna(row.display_name) else row.name

        try:
            df_tbl = labs_wh.get_warehouse_tables(warehouse=warehouse_id, workspace=workspace_id)
            if not isinstance(df_tbl, pd.DataFrame):
                df_tbl = pd.DataFrame(df_tbl)
            if not df_tbl.empty:
                df_tbl['warehouse_id'] = warehouse_id
                df_tbl['warehouse_name'] = warehouse_name
                df_tbl['workspace_id'] = workspace_id
                warehouse_table_frames.append(create_pk(df_tbl, ['Table Name', 'warehouse_id'], 'warehouse_table_pk'))

            df_col = labs_wh.get_warehouse_columns(warehouse=warehouse_id, workspace=workspace_id)
            if not isinstance(df_col, pd.DataFrame):
                df_col = pd.DataFrame(df_col)
            if not df_col.empty:
                df_col['warehouse_id'] = warehouse_id
                df_col['warehouse_name'] = warehouse_name
                df_col['workspace_id'] = workspace_id
                warehouse_column_frames.append(create_pk(df_col, ['Table Name', 'Column Name', 'warehouse_id'], 'warehouse_column_pk'))
        except Exception as e:
            print(f"Error processing warehouse {warehouse_name if pd.notna(warehouse_name) else warehouse_id} in workspace {workspace_id}: {e}")

    df_warehouse_tables = pd.concat(warehouse_table_frames, axis=0, ignore_index=True) if warehouse_table_frames else pd.DataFrame()
    df_warehouse_columns = pd.concat(warehouse_column_frames, axis=0, ignore_index=True) if warehouse_column_frames else pd.DataFrame()

    write_raw_table(df_warehouse_tables, 't_warehouse_tables')
    write_raw_table(df_warehouse_columns, 't_warehouse_columns')

    print(
        f"Lakehouse/Warehouse extraction complete: lakehouse_tables={len(df_lakehouse_tables)}, "
        f"lakehouse_columns={len(df_lakehouse_columns)}, lakehouse_shortcuts={len(df_lakehouse_shortcuts)}, "
        f"warehouse_tables={len(df_warehouse_tables)}, warehouse_columns={len(df_warehouse_columns)}"
    )
else:
    print('Skipping warehouse table/column extraction (artifactTypes filter).')
end_phase('warehouse_structure')

StatementMeta(, dd093f86-5ee4-4919-805d-03c5b34cf858, 12, Finished, Available, Finished, False)

# DirectLake

In [ ]:
# Debug-only cell disabled for production extraction runs.
# df_shortcuts = labs.lakehouse.list_shortcuts(lakehouse='<lakehouse_id>', workspace='<workspace_id>')
# df_shortcuts

StatementMeta(, dd093f86-5ee4-4919-805d-03c5b34cf858, 11, Finished, Available, Finished, False)

,Shortcut Name,Shortcut Path,Source Type,Source Workspace Id,Source Workspace Name,Source Item Id,Source Item Name,Source Item Type,OneLake Path,Connection Id,Location,Bucket,SubPath,Source Properties Raw
0,Medallion,/Tables,OneLake,6feccad2-02d1-4257-b15f-c875f8f7e71b,Data Product,1ad80ec0-8d27-46e7-bc81-b5aeb27b7162,WH_SampleData,Warehouse,None,None,None,None,None,"{'type': 'OneLake', 'oneLake': {'workspaceId':..."
1,HackneyLicense,/Tables,OneLake,6feccad2-02d1-4257-b15f-c875f8f7e71b,Data Product,1ad80ec0-8d27-46e7-bc81-b5aeb27b7162,WH_SampleData,Warehouse,None,None,None,None,None,"{'type': 'OneLake', 'oneLake': {'workspaceId':..."
2,Geography,/Tables,OneLake,6feccad2-02d1-4257-b15f-c875f8f7e71b,Data Product,1ad80ec0-8d27-46e7-bc81-b5aeb27b7162,WH_SampleData,Warehouse,None,None,None,None,None,"{'type': 'OneLake', 'oneLake': {'workspaceId':..."
3,Date,/Tables,OneLake,6feccad2-02d1-4257-b15f-c875f8f7e71b,Data Product,1ad80ec0-8d27-46e7-bc81-b5aeb27b7162,WH_SampleData,Warehouse,None,None,None,None,None,"{'type': 'OneLake', 'oneLake': {'workspaceId':..."


In [ ]:
# Debug-only cell disabled for production extraction runs.
# df_semantic_models.head()

StatementMeta(, 3f03ec58-ec94-4b6f-bf8b-f79ee66945dc, 15, Finished, Available, Finished, False)

,id,display_name,description,type,workspace_id,folder_id
5,a1b49376-ff7c-4042-99cc-ddbb6fb49222,Corporate Spend,,SemanticModel,6feccad2-02d1-4257-b15f-c875f8f7e71b,None
6,b1efbb19-9369-4df3-a4b8-fa4db7e7a443,Customer Profitability Sample PBIX (1),,SemanticModel,6feccad2-02d1-4257-b15f-c875f8f7e71b,None
7,02f5c715-4d75-4136-a130-6591532a80cf,SemanticModel Trips,,SemanticModel,6feccad2-02d1-4257-b15f-c875f8f7e71b,None
8,84c4c351-7cdc-4c86-859e-f5752f07160b,DirectLakeWarehouse,,SemanticModel,6feccad2-02d1-4257-b15f-c875f8f7e71b,None
9,f9886f9f-4dbe-446f-8fbc-b2a1c46c8ec2,DirectLake Shortcut Data,,SemanticModel,6feccad2-02d1-4257-b15f-c875f8f7e71b,None


In [ ]:
start_phase('directlake')
if directlake_requested:
    direct_lake_frames = []
    for row in df_semantic_models[['id', 'workspace_id']].itertuples(index=False):
        dataset_id = row.id
        workspace_id = row.workspace_id
        try:
            df_direct = labs.directlake.get_direct_lake_sources(dataset=dataset_id, workspace=workspace_id)
            if not isinstance(df_direct, pd.DataFrame):
                df_direct = pd.DataFrame(df_direct)
            if not df_direct.empty:
                df_direct['dataset_id'] = dataset_id
                df_direct['workspace_id'] = workspace_id
                direct_lake_frames.append(create_pk(df_direct, ['itemName', 'dataset_id'], 'direct_lake_pk'))
        except Exception as e:
            print(f"Error processing direct lake sources for dataset {dataset_id} in workspace {workspace_id}: {e}")

    df_directLake = pd.concat(direct_lake_frames, axis=0, ignore_index=True) if direct_lake_frames else pd.DataFrame()
    write_raw_table(df_directLake, 't_direct_lake_sources')
    print(f"DirectLake extraction complete: rows={len(df_directLake)}")
else:
    print('Skipping directlake extraction (artifactTypes filter).')
end_phase('directlake')

StatementMeta(, 3f03ec58-ec94-4b6f-bf8b-f79ee66945dc, 16, Finished, Available, Finished, False)

In [ ]:
notebookutils.session.stop()

StatementMeta(, dd093f86-5ee4-4919-805d-03c5b34cf858, 13, Finished, Available, Finished, False)